In [1]:
# --------------------------------------------------------
# External packages
from jax import grad, config, jit
config.update("jax_enable_x64", True)
import equinox as eqx
import jax.numpy as jnp
import matplotlib.pyplot as plt

# --------------------------------------------------------
# KiRATE modules
from KiRATE import constants
from KiRATE.kinetics import Arrhenius

Mathematical derivation of $dk / dT$ for Arrhenius equation.
    
Given: $k(T) = A \times T^n \times exp(-Ea/(R \times T))$
    
Using product rule: $dk/dT = f'*g*h + f*g'*h + f*g*h'$
    
Where:
- $f = A$ (constant, so $f' = 0$)
- $g = T^n$ enche $g' = n \times T^{(n-1)}$
- $h = exp(-Ea/(R \times T))$ (so $h' = exp(-Ea/(R \times T)) \times Ea/(R \times T^2)$)
    
Therefore:
\begin{align}
dk/dT &= A \times \left[n \times T^{(n-1)} \times exp(-Ea/(R \times T)) + T^n \times exp(-Ea/(R \times T)) \times Ea/(R \times T^2)\right] \\
      &= A \times exp(-Ea/(R \times T)) \times [n \times T^{(n-1)} + T^n \times Ea/(R \times T^2)]\\
      &= k(T) \times [n/T + Ea/(R \times T^2)]
\end{align}

In [9]:
reaction = Arrhenius(
    name="H2+O=H+OH",
    parameters={"A": 5.080e+04, "n": 2.670E+00, "Ea": 6.292e+03}
)

T = jnp.linspace(300.0, 3000, 10)
T_test = 300
T_jax = jnp.asarray(T_test, dtype=jnp.float64)
# auto_dkdT = reaction.grad_temperature(T)
auto_dkdTheta = reaction.grad_params(T)

print(auto_dkdTheta.A, auto_dkdTheta.n, auto_dkdTheta.Ea)

[1.07222723e+02 1.33613512e+05 2.29061373e+06 1.18989105e+07
 3.65963926e+07 8.46524962e+07 1.64256162e+08 2.83276567e+08
 4.49209953e+08 6.69192490e+08] [3.10680144e+07 4.34195848e+10 7.91548271e+11 4.28570083e+12
 1.35959842e+13 3.22334299e+13 6.38306649e+13 1.12004093e+14
 1.80299919e+14 2.72176293e+14] [-9.13664560e+03 -5.69272666e+06 -6.50624711e+07 -2.53482017e+08
 -6.23689197e+08 -1.20223251e+09 -1.99951000e+09 -3.01731472e+09
 -4.25311190e+09 -5.70231240e+09]


In [ ]:
reaction = Arrhenius(
    name="H2+O=H+OH",
    parameters={"A": 5.080e+04, "n": 2.670E+00, "Ea": 6.292e+03}
)

# --------------------------------------------------------
# Compute the gradient of the rate constant w.r.t
# the Temperature this function below is the analytical
# derivative of it.
def dkdT_analytical(reaction_instance, T):
    """
    Analytical form of the dk/dT
    """
    k = reaction_instance.rate_constant(T)
    n = reaction_instance.n
    Ea = reaction_instance.Ea
    return k * (n/T + Ea/(constants.R_cal_mol * T**2))

# JAX automatic differentiation
dkdT_autodiff = grad(reaction.rate_constant)

print(f"Analytical derivative evaluated at 300 K: {dkdT_analytical(reaction, 300.0):.18f}")
print(f"Autodiff   derivative evaluated at 300 K: {dkdT_autodiff(300.0):.18f}")
print(f"Autodiff   derivative evaluated at 300 K: {reaction.grad_T(300.0):.18f}")

In [ ]:
def analytical_gradients(reaction_instance, temperature):
    A, n, Ea = reaction_instance.A, reaction_instance.n, reaction_instance.Ea
        
    # Common terms for efficiency
    T_n = jnp.power(temperature, n)
    exp_term = jnp.exp(-Ea / (constants.R_cal_mol * temperature))
    k = reaction_instance.rate_constant(temperature)  # Rate constant
        
    # Analytical gradients
    dkdA = T_n * exp_term
    dkdn = A * T_n * jnp.log(temperature) * exp_term
    dkdEa = -k / (constants.R_cal_mol * temperature)
        
    return dkdA, dkdn, dkdEa

T = 300.0
auto_grad = reaction.grad_params(T)
dkdA_auto, dkdn_auto, dkdEa_auto = auto_grad.A, auto_grad.n, auto_grad.Ea
dkdA_anal, dkdn_anal, dkdEa_anal = analytical_gradients(reaction, T)

print("=== Gradient Comparison ===")
print(f"dk/dA  - Auto:  {dkdA_auto:.12e}, Analytical:  {dkdA_anal:.12e}")
print(f"dk/dn  - Auto:  {dkdn_auto:.12e}, Analytical:  {dkdn_anal:.12e}")
print(f"dk/dEa - Auto: {dkdEa_auto:.12e}, Analytical: {dkdEa_anal:.12e}")

In [ ]:
T = jnp.linspace(300.0, 3000, 10)
auto_grad = reaction.grad_params(T)

In [ ]:
# def autodiff_gradients(reaction_instance, temperature):
#     wrapper_function = lambda model: model.rate_constant(temperature)
#     eqx_gradients = grad(wrapper_function)(reaction_instance)
#     return eqx_gradients.A, eqx_gradients.n, eqx_gradients.Ea
def autodiff_gradients(reaction_instance, temperature):
    @jit
    def gradient_fn(model, T):
        wrapper_function = lambda m: m.rate_constant(T)
        return grad(wrapper_function)(model)
    
    eqx_gradients = gradient_fn(reaction_instance, temperature)
    return eqx_gradients.A, eqx_gradients.n, eqx_gradients.Ea

def analytical_gradients(reaction_instance, temperature):
    A, n, Ea = reaction_instance.A, reaction_instance.n, reaction_instance.Ea
        
    # Common terms for efficiency
    T_n = jnp.power(temperature, n)
    exp_term = jnp.exp(-Ea / (constants.R_cal_mol * temperature))
    k = reaction_instance.rate_constant(temperature)  # Rate constant
        
    # Analytical gradients
    dkdA = T_n * exp_term
    dkdn = A * T_n * jnp.log(temperature) * exp_term
    dkdEa = -k / (constants.R_cal_mol * temperature)
        
    return dkdA, dkdn, dkdEa

# Now compare both methods
T = 300.0
dkdA_auto, dkdn_auto, dkdEa_auto = autodiff_gradients(reaction, T)
dkdA_anal, dkdn_anal, dkdEa_anal = analytical_gradients(reaction, T)

print("=== Gradient Comparison ===")
print(f"dk/dA  - Auto:  {dkdA_auto:.12e}, Analytical:  {dkdA_anal:.12e}")
print(f"dk/dn  - Auto:  {dkdn_auto:.12e}, Analytical:  {dkdn_anal:.12e}")
print(f"dk/dEa - Auto: {dkdEa_auto:.12e}, Analytical: {dkdEa_anal:.12e}")






@eqx.filter_grad
def parameter_gradients(model, T):  # Model is first argument
    return model.rate_constant(T)

result = parameter_gradients(reaction, 300.0)
print(f"dkdA: {result.A}")
print(f"dkdn: {result.n}")
print(f"dkdEa: {result.Ea}")


@eqx.filter_grad
def parameter_gradients(model, T):  # Model is first argument
    return model.rate_constant(T)




print(f"\nAbsolute Errors:")
print(f"dk/dA:  {abs(dkdA_auto - dkdA_anal):.10e}")
print(f"dk/dn:  {abs(dkdn_auto - dkdn_anal):.10e}")
print(f"dk/dEa: {abs(dkdEa_auto - dkdEa_anal):.10e}")

# Test at multiple temperatures
print(f"\n=== Multiple Temperature Test ===")
temperatures = [300, 500, 1000, 1500, 2000]
for T_test in temperatures:
    auto_grads = autodiff_gradients(reaction, T_test)
    anal_grads = analytical_gradients(reaction, T_test)
    
    errors = [abs(auto_grads[i] - anal_grads[i]) for i in range(3)]
    max_error = max(errors)
    
    print(f"T={T_test:4d} K: Max error = {max_error:.2e}")